# EDA corto: ¿Se pueden anticipar los períodos de baja disponibilidad de embalses en Colombia?
Datos diarios 2015-2026: demanda eléctrica, nivel de embalses y El Niño (índice ONI).

In [ ]:
# Sube el archivo base_diaria_integrada.csv a Colab (ejecuta y elige el archivo)
from google.colab import files
files.upload()

## 1. Cargar y revisar los datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
RUTA = "base_diaria_integrada.csv"

df = pd.read_csv(RUTA, parse_dates=["fecha"]).set_index("fecha")

print("Filas y columnas:", df.shape)
print("Desde", df.index.min().date(), "hasta", df.index.max().date())
print("Días faltantes en el calendario:", len(pd.date_range(df.index.min(), df.index.max()).difference(df.index)))
print("\nDatos vacíos por columna:")
print(df.isna().sum()[df.isna().sum() > 0])

## 2. Definir "baja disponibilidad" (nuestra variable objetivo)

In [ ]:
# Idea simple: un día es "bajo" si el llenado está en el 20% más bajo de toda la historia.
nivel = df["emb_llenado_pct"]
umbral = nivel.quantile(0.20)
df["nivel_bajo"] = (nivel < umbral).astype(int)
df.loc[nivel.isna(), "nivel_bajo"] = np.nan

print(f"Umbral de nivel bajo: {umbral:.1f}% de llenado")
print(f"Porcentaje de días en nivel bajo: {df['nivel_bajo'].mean()*100:.1f}%")

## 3. Nivel de los embalses en el tiempo

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
nivel.plot(ax=ax, color="steelblue", lw=1)
ax.axhline(umbral, color="red", ls="--", label=f"Umbral bajo ({umbral:.0f}%)")
ax.fill_between(df.index, nivel, umbral, where=nivel < umbral, color="red", alpha=0.3)
ax.set_title("Llenado de embalses (%)")
ax.set_ylabel("% de llenado")
ax.legend()
plt.tight_layout()
plt.show()

## 4. ¿Hay una época del año más crítica? (estacionalidad)

In [ ]:
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
prom_mes = df.groupby(df.index.month)["emb_llenado_pct"].mean()
bajo_mes = df.groupby(df.index.month)["nivel_bajo"].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(meses, prom_mes.values, color="steelblue")
axes[0].set_title("Llenado promedio por mes (%)")
axes[1].bar(meses, bajo_mes.values, color="indianred")
axes[1].set_title("% de días en nivel bajo, por mes")
plt.tight_layout()
plt.show()

## 5. Demanda eléctrica: tendencia y patrón semanal

In [ ]:
dem_mensual = df["dem_total_mwh"].resample("MS").mean() / 1000

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
dem_mensual.plot(ax=axes[0], color="darkorange")
axes[0].set_title("Demanda diaria promedio por mes (GWh)")

dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
(df.groupby("dia_semana")["dem_total_mwh"].mean() / 1000).plot.bar(ax=axes[1], color="darkorange")
axes[1].set_xticklabels(dias, rotation=0)
axes[1].set_title("Demanda promedio por día de la semana (GWh)")
plt.tight_layout()
plt.show()

## 6. El Niño y los embalses

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
oni_mensual = df["oni"].resample("MS").mean()
ax.plot(oni_mensual.index, oni_mensual, color="black", lw=1)
ax.fill_between(oni_mensual.index, oni_mensual, 0, where=oni_mensual >= 0.5, color="red", alpha=0.4, label="El Niño (calor, menos lluvia)")
ax.fill_between(oni_mensual.index, oni_mensual, 0, where=oni_mensual <= -0.5, color="blue", alpha=0.4, label="La Niña (más lluvia)")
ax.set_title("Índice ONI (clima del Pacífico)")
ax.legend()
plt.tight_layout()
plt.show()

print("Llenado promedio según el clima:")
print(df.groupby("elnino")["emb_llenado_pct"].mean().rename({0: "Sin El Niño", 1: "Con El Niño"}).round(1))

## 7. ¿Qué tanto "se adelanta" el clima y la demanda al nivel de los embalses?

In [ ]:
# Comparamos el llenado de hoy con el ONI y la demanda de meses atrás (datos mensuales).
m = df.resample("MS").mean(numeric_only=True)
lags = range(0, 9)
corr_oni = [m["emb_llenado_pct"].corr(m["oni"].shift(l)) for l in lags]
corr_dem = [m["emb_llenado_pct"].corr(m["dem_total_mwh"].shift(l)) for l in lags]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lags, corr_oni, "o-", label="ONI (clima)", color="crimson")
ax.plot(lags, corr_dem, "s-", label="Demanda", color="darkorange")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("Meses de anticipación")
ax.set_ylabel("Correlación con el llenado actual")
ax.set_title("¿Cuánto ayuda mirar hacia atrás?")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Línea base: ¿el nivel de hoy ya predice el de dentro de 30 días?

In [ ]:
# Si esto ya funciona muy bien, hay que demostrar que clima y demanda aportan algo extra.
futuro = nivel.shift(-30)
print("Correlación entre el llenado de hoy y el de dentro de 30 días:", round(nivel.corr(futuro), 2))
print("Correlación entre el ONI de hoy y el llenado de dentro de 30 días:", round(df["oni"].corr(futuro), 2))